# Real-Time Mode in Structured Streaming

All the queries in this chapter so far ran as **micro-batches**: collect the
rows that arrived, process them as a small batch job, commit, repeat. That
is efficient and easy to recover, but every batch pays a fixed overhead, so
latency cannot drop below a certain floor. **Real-time mode** is a
Structured Streaming execution mode that removes that floor. It processes
records continuously as they arrive, with end-to-end latency as low as a few
milliseconds, while keeping the same DataFrame API. It was added to Apache
Spark in version 4.1.0 and is available on Databricks classic compute.

This notebook explains how real-time mode works, how to write and size a
real-time query, and when it is worth its cost. Real-time mode does not run
on serverless compute, which this course uses, so most code on this page is
shown rather than executed.

## The Latency Floor of Micro-Batches

{doc}`Introduction to Stream Processing <01_introduction_to_stream_processing>`
compared the two processing models. Micro-batching achieves high throughput
and simple recovery because each batch is a self-contained unit of work. But
each batch also has to be planned, scheduled as tasks, written to the sink
and committed to the checkpoint before the next one starts. The progress
report in
{doc}`Spark Structured Streaming <02_spark_structured_streaming>` broke a
batch duration down into exactly these phases: `queryPlanning`,
`latestOffset`, `addBatch`, `walCommit`, `commitBatch` and so on. Even for a
few hundred rows, a micro-batch took seconds there.

A record that arrives just after a batch started has to wait for that batch
to finish, and then for the whole next batch. For a dashboard that refreshes
every minute this does not matter. For fraud detection, where a payment must
be approved or blocked while the customer waits, it does.

## How Real-Time Mode Works

Real-time mode changes three things about how a query executes:

- **Long-running batches.** Instead of short batches that each read a fixed
  set of new records, a batch runs for a fixed duration, five minutes by
  default, and processes records continuously as they arrive in the source
  during that time.
- **Concurrent stages.** In micro-batch mode, a stage runs after the stage
  before it has finished. In real-time mode, all stages of the query are
  scheduled at once and stay running for the whole batch.
- **Streaming shuffle.** Data moves from one stage to the next as soon as it
  is produced, instead of being written out in full before the next stage
  reads it.

```mermaid
:caption: In micro-batch mode, each record waits for its batch to be planned, executed and committed. In real-time mode, the stages run side by side and a record flows through them as soon as it arrives. Progress is committed only between long-running batches.
:align: center

graph LR
    subgraph Micro-batch
        M1["Plan batch"] --> M2["Run stage 1"] --> M3["Run stage 2"] --> M4["Commit"] --> M1
    end
    subgraph Real-time
        R1(["Record"]) --> R2["Stage 1, always running"] --> R3["Stage 2, always running"] --> R4(["Sink"])
    end
```

Checkpointing still happens, but only at the end of each long-running batch.
The batch duration is therefore not a latency setting but a trade-off
between checkpoint frequency and recovery time: a longer batch means less
overhead from checkpointing, but more data to reprocess after a failure, and
metrics that are published less often.

## Requirements on Databricks

Real-time mode places strict requirements on the compute it runs on:

| Setting | Requirement |
| --- | --- |
| Compute | Classic compute. Serverless compute is not supported. |
| Access mode | Dedicated, or Standard for Python queries only |
| Databricks Runtime | 16.4 LTS or above |
| Autoscaling | Off |
| Photon | Off |
| Spot instances | Off, so that executors are not interrupted |
| Spark configuration | `spark.databricks.streaming.realTimeMode.enabled true` |

The cluster runs all the time, and every stage of the query permanently
occupies task slots. That makes a real-time query considerably more
expensive than a scheduled `availableNow` job processing the same data.

## Writing a Real-Time Query

A real-time query is an ordinary streaming query with two changes: the
trigger is `realTime`, and the output mode must be `update`. The duration
passed to the trigger is the length of the long-running batch, and Python
requires it to be given explicitly.

The example below reads bakery sales from a Kafka topic, flags large orders
and publishes them to a second topic, where a service that alerts the
bakery staff is listening. It only uses operations that real-time mode
supports: parsing, filtering, a projection and a broadcast join with a small
static table.

```python
from pyspark.sql import functions as F

SALES_SCHEMA = """
    transactionID BIGINT, customerID BIGINT, franchiseID BIGINT, dateTime TIMESTAMP,
    product STRING, quantity BIGINT, unitPrice BIGINT, totalPrice BIGINT, paymentMethod STRING
"""

franchises = spark.read.table("samples.bakehouse.sales_franchises").select("franchiseID", "name", "city")

large_orders = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker-1:9092,broker-2:9092")
    .option("subscribe", "bakery-sales")
    .option("maxPartitions", 8)
    .load()
    .select(F.from_json(F.col("value").cast("string"), SALES_SCHEMA).alias("sale"))
    .select("sale.*")
    .filter(F.col("quantity") >= 30)
    .join(F.broadcast(franchises), "franchiseID")
    .select(
        F.col("transactionID").cast("string").alias("key"),
        F.to_json(F.struct("transactionID", "name", "city", "product", "quantity", "dateTime")).alias("value"),
    )
)

query = (
    large_orders.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker-1:9092,broker-2:9092")
    .option("topic", "large-orders")
    .option("checkpointLocation", "/Volumes/main/bakery/checkpoints/large_orders")
    .outputMode("update")
    .trigger(realTime="5 minutes")
    .start()
)
```

Apart from the trigger and the output mode, this is the same code you would
write for a micro-batch query. Switching an existing query between the two
modes is mostly a matter of changing those two lines.

### Trying It on Serverless Compute

Starting the same kind of query on this course's serverless compute shows
the compute requirement in action. The query is rejected before it ever
contacts a Kafka broker:

In [1]:
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

try:
    (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", "broker-1:9092")
        .option("subscribe", "bakery-sales")
        .load()
        .writeStream.format("kafka")
        .option("kafka.bootstrap.servers", "broker-1:9092")
        .option("topic", "large-orders")
        .option("checkpointLocation", "/Volumes/main/bakery/checkpoints/large_orders")
        .outputMode("update")
        .trigger(realTime="5 minutes")
        .start()
    )
except Exception as e:
    print(type(e).__name__, "-", str(e).splitlines()[0])

AnalysisException - [UC_COMMAND_NOT_SUPPORTED_IN_SHARED_ACCESS_MODE] The command(s): Real Time Mode (RealTimeTrigger) are not supported for Unity Catalog clusters in shared access mode. Use single user access mode instead. SQLSTATE: 0AKUC


Serverless compute runs in a shared, standard access mode, which does not
support the real-time trigger. To run the example for real, use a classic
cluster that meets the requirements listed above.

## What Real-Time Mode Supports

Because records flow through continuously, not every source, sink and
operation can take part. On Databricks, the current support is as follows.

| | Supported |
| --- | --- |
| Sources | Apache Kafka, Azure Event Hubs (through the Kafka connector), Amazon Kinesis (EFO mode recommended), Amazon MSK |
| Sinks | Apache Kafka, Azure Event Hubs (through the Kafka connector), custom sinks with `foreach` |
| Stateless operations | Selections, projections, `union`, Scala and Python UDFs |
| Stateful operations | Aggregations such as `count`, `sum` and `avg`; tumbling and sliding windows; `dropDuplicates` and `dropDuplicatesWithinWatermark`; `transformWithState` |
| Joins | Stream-static joins, broadcast when the table is 10 MB or smaller; stream-stream inner joins on Databricks Runtime 18 and above, with extra configuration |
| Output mode | `update` only |

Notable things that are **not** supported:

- **Delta tables**, as either source or sink. The files, Auto Loader and
  Delta tables used elsewhere in this chapter belong to micro-batch
  streaming. If the data needs to land in a Delta table as well, run a
  second, micro-batch query from the same Kafka topic.
- **`foreachBatch`**, because there are no small batches to hand to it. Use
  `foreach` for custom sinks.
- Session windows, stream-stream outer joins, and stream-static joins
  without a broadcast.
- The `append` and `complete` output modes.

Open-source Apache Spark started with a narrower set: Spark 4.1.0 supports
stateless queries reading from Kafka, with stateful operations following in
later releases. Check the documentation of your Spark version or Databricks
Runtime before relying on a particular operation.

## Sizing a Real-Time Cluster

In micro-batch mode, a cluster with fewer task slots than tasks still works:
the tasks just run in several waves, as described in
{doc}`Distributed Systems Programming Fundamentals <../01_introduction_to_apache_spark/03_distributed_systems_programming_fundamentals/03_distributed_systems_programming_fundamentals>`.
In real-time mode, every stage runs at the same time for the whole batch,
so **every task needs its own slot from the start**. The number of available
task slots must be at least the total number of tasks across all stages of
the query. A query that does not fit fails to start.

:::::{dropdown} Example: How Many Task Slots Does a Query Need?
:open:
:color: primary
:icon: pencil
:class-body: example-body

A real-time query reads from Kafka with `maxPartitions` set to 8. How many
task slots does it need, first as a stateless pipeline and then with an
aggregation added?

**Stateless: Kafka to Kafka, as in the example above.** The query has a
single stage, with one task per source partition:

$$
8 \text{ slots}
$$

**Add a `groupBy` aggregation.** The aggregation needs a shuffle, which adds
a second stage with one task per shuffle partition. With
`spark.sql.shuffle.partitions` set to 20, both stages run concurrently:

$$
8 + 20 = 28 \text{ slots}
$$

On a cluster of workers with 4 cores each, the stateless query fits on 2
workers, while the aggregating query needs $\lceil 28 / 4 \rceil = 7$
workers. Lowering `spark.sql.shuffle.partitions` to match the throughput
you actually need is the easiest way to keep a stateful real-time query
affordable.
:::::

When `maxPartitions` is not set, the number of source tasks equals the
number of partitions of the Kafka topic.

## Monitoring Latency

In real-time mode, batch durations no longer describe latency, because a
batch lasts minutes by design. Instead, each progress report contains a
`latencies` object with three metrics, each reported as percentiles from P0
to P99:

| Metric | Measures the time from | To |
| --- | --- | --- |
| `sourceQueuingLatencyMs` | A record being written to the message bus | The query reading it |
| `processingLatencyMs` | The query reading a record | Writing it downstream |
| `e2eLatencyMs` | A record being written to the message bus | The query writing it downstream |

High queuing latency points to the source or to a query that cannot keep up.
High processing latency points to an expensive operation inside the query.
The reports are available through `query.lastProgress`, a
`StreamingQueryListener`, or the streaming dashboard in the Databricks UI,
and are published at the end of each long-running batch.

## Delivery Guarantees

Real-time mode keeps Structured Streaming's fault-tolerance model: progress
is checkpointed between batches, and after a failure the query resumes from
the last checkpoint. Processing inside Spark, including state, is
exactly-once. The supported sinks, Kafka and `foreach`, are **at-least-once**,
though: records written to them after the last checkpoint are written again
after a restart. Consumers of a real-time query's output should therefore
tolerate duplicates, for example by de-duplicating on a key such as
`transactionID`, which the example above uses as the Kafka message key.

## Choosing an Execution Mode

| Latency the business needs | Trigger | Compute |
| --- | --- | --- |
| Hours, or minutes | `availableNow`, scheduled as a job | Runs only while processing; serverless works |
| Seconds to a minute | `processingTime`, or the default trigger | Always running |
| Below a second | `realTime` | Always running; dedicated classic cluster, sized for all stages at once |

Start from the latency the use case actually needs, not the lowest one
available. Most pipelines that feed tables, dashboards and reports are best
served by scheduled `availableNow` runs, as in the rest of this chapter.
Real-time mode is for the operational cases, such as fraud detection,
personalisation and alerting, where a decision has to be made while the
event is still happening.

## Resources

- [Real-time mode in Structured Streaming (Databricks)](https://docs.databricks.com/aws/en/structured-streaming/real-time)
- [Set up real-time mode (Databricks)](https://docs.databricks.com/aws/en/structured-streaming/real-time/setup)
- [Real-time mode reference (Databricks)](https://docs.databricks.com/aws/en/structured-streaming/real-time/reference)
- [Optimize and monitor real-time mode query performance (Databricks)](https://docs.databricks.com/aws/en/structured-streaming/real-time/performance)
- [Breaking the micro-batch barrier: the architecture of Apache Spark Real-Time Mode (Databricks blog)](https://www.databricks.com/blog/breaking-microbatch-barrier-architecture-apache-spark-real-time-mode)